In [0]:
-- 1. Parse JSON, Type, and Trim (Enabled for Streaming)
CREATE TEMPORARY STREAMING LIVE VIEW stg_inventory_typed AS
SELECT
  TRIM(payload.event_id) AS event_id,
  
  -- Handle date formatting anomalies
  COALESCE(
    try_to_date(TRIM(payload.restock_date), 'M/d/yyyy'),
    try_to_date(TRIM(payload.restock_date), 'yyyy-M-d'),
    try_to_date(TRIM(payload.restock_date), 'd-MMM-yy'),
    CAST(TRIM(payload.restock_date) AS DATE)
  ) AS restock_date,
  
  CAST(TRIM(payload.product_id) AS INT) AS product_id,
  CAST(TRIM(payload.store_id) AS INT) AS store_id,
  CAST(TRIM(payload.restock_quantity) AS INT) AS restock_quantity,
  
  event_timestamp AS _ingestion_timestamp
FROM (
  SELECT 
    from_json(
      message_payload, 
      'event_id STRING, restock_date STRING, product_id STRING, store_id STRING, restock_quantity STRING'
    ) AS payload,
    event_timestamp
  FROM STREAM(mavenmarket.bronze.kafka_inventory)
);

-- 2. Run Generic KPI Checks (Completeness, Validity, and Stream-Static RI)
CREATE TEMPORARY STREAMING LIVE VIEW stg_inventory_checked AS
SELECT 
  i.*,
  
  -- HARD CHECKS (Route to Quarantine)
  CASE WHEN i.event_id IS NULL THEN 'Completeness: event_id is null; ' ELSE '' END AS err_event_id,
  CASE WHEN i.restock_quantity IS NULL THEN 'Completeness: restock_quantity is null; ' ELSE '' END AS err_qty_null,
  CASE WHEN i.restock_quantity <= 0 THEN 'Validity: restock_quantity must be greater than 0; ' ELSE '' END AS err_qty_neg,
  
  -- Referential Integrity: Stream-Static Join against dimension tables
  CASE WHEN i.product_id IS NOT NULL AND p.product_id IS NULL 
       THEN 'Referential Integrity: product_id not found in products; ' ELSE '' END AS err_fk_product,
  CASE WHEN i.store_id IS NOT NULL AND s.store_id IS NULL 
       THEN 'Referential Integrity: store_id not found in stores; ' ELSE '' END AS err_fk_store,
       
  -- WARN CHECKS (Stay in Silver, added to dq_flags)
  CASE WHEN i.restock_date IS NULL THEN 'Completeness: restock_date is null; ' ELSE '' END AS warn_date
FROM STREAM(live.stg_inventory_typed) i
-- Perform a stream-static join to the Bronze static tables to validate foreign keys
LEFT JOIN mavenmarket.bronze.products p 
  ON i.product_id = CAST(p.product_id AS INT)
LEFT JOIN mavenmarket.bronze.stores s 
  ON i.store_id = CAST(s.store_id AS INT);

-- 3. Consolidate Flags
CREATE TEMPORARY STREAMING LIVE VIEW stg_inventory_flagged AS
SELECT 
  *,
  TRIM(CONCAT(err_event_id, err_qty_null, err_qty_neg, err_fk_product, err_fk_store)) AS hard_failures,
  TRIM(CONCAT(warn_date)) AS dq_flags
FROM STREAM(live.stg_inventory_checked);

-- 4. Write Target Tables (Split the stream)
CREATE STREAMING LIVE TABLE silver_inventory
COMMENT "Cleaned Kafka inventory restocks with warn-level data quality flags."
AS SELECT 
  * EXCEPT(err_event_id, err_qty_null, err_qty_neg, err_fk_product, err_fk_store, warn_date, hard_failures)
FROM STREAM(live.stg_inventory_flagged)
WHERE hard_failures = '';

CREATE STREAMING LIVE TABLE quarantine_inventory
COMMENT "Quarantined inventory failing hard checks"
AS SELECT 
  * EXCEPT(err_event_id, err_qty_null, err_qty_neg, err_fk_product, err_fk_store, warn_date, dq_flags),
  hard_failures AS quarantine_reasons
FROM STREAM(live.stg_inventory_flagged)
WHERE hard_failures != '';